# Managing messages — summarize and trim the conversation
**by Kassouar Fatima 🇩🇿 — Module 3 project**

Keep a growing conversation small: **`SummarizationMiddleware`** collapses old turns into a summary as soon as the history crosses the token threshold, and a custom **`before_agent`** hook deletes messages from state before each run. Neither touches the tools or the prompt — only what the model will see next.

- **Run:** double-click `run_3_2.cmd` in this folder → *Run → Run All Cells*.
- **Needs:** `GOOGLE_API_KEY` from the `.env` next to this notebook.
- **Model:** `gemini-3.5-flash-lite` in both slots (agent + summarizer) — its own 500/day bucket, so it runs even while Modules 1–2's `gemini-3.1-flash-lite` bucket is spent. **Model fix:** the course used `gpt-5-nano` → Gemini here, no extra key.

**Idea:** whatever rewrites state rewrites what the model sees — middleware is the sanctioned place to do it, around every turn, invisibly to the agent.

**What I learned:**
- The config is the policy: `trigger=("tokens", 100)` fires, `keep=("messages", 1)` keeps the newest message and summarizes the rest.
- The summarizer is a model call too — it needs its own `model=`.
- Trim by **ids**, not content: collect the ids first (here every `ToolMessage`), then `RemoveMessage(id=...)`.
- Both need a checkpointer (`InMemorySaver()`) — without one there is no state to summarize or trim.


## Setup

In [1]:
import importlib.util as _ilu
import subprocess as _sp
import sys as _sys

# Install only what is missing. Uses pip when available, falls back to uv
# (the local venv on this machine is created by uv and has no pip).
REQUIRED = {
    "langchain": "langchain",
    "langgraph": "langgraph",
    "langchain_google_genai": "langchain-google-genai",
    "langchain_community": "langchain-community",
    "dotenv": "python-dotenv",
}
missing = [pkg for imp, pkg in REQUIRED.items() if _ilu.find_spec(imp) is None]

if missing:
    print("Installing:", ", ".join(missing))
    if _sp.run([_sys.executable, "-m", "pip", "install", "-q", *missing]).returncode != 0:
        _sp.run(["uv", "pip", "install", *missing])
else:
    print("All packages already installed.")

All packages already installed.


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # local run: finds the .env above the notebook folder

# Colab run: read the key from the Secrets panel (key icon in the left sidebar)
try:
    from google.colab import userdata
    try:
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
except ImportError:
    pass

def is_real(value):
    # empty values and template placeholders ("your_..." ) count as missing
    return bool(value) and not value.startswith("your_")

# Accept both names: the local project keeps the key as GEMINI_API_KEY
for name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
    if is_real(os.getenv(name)):
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = os.getenv(name)
        break

# Still missing? Paste it here (hidden input).
if not is_real(os.getenv("GOOGLE_API_KEY")):
    try:
        value = getpass("Gemini API key is missing - paste it (Enter to skip): ").strip()
    except Exception:
        value = ""
    if value:
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = value

print("GOOGLE_API_KEY:", "set" if is_real(os.getenv("GOOGLE_API_KEY")) else "MISSING")

# Tracing with a placeholder LangSmith key only produces 401/403 warnings: switch it off
if not is_real(os.getenv("LANGSMITH_API_KEY")):
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: off (no valid LANGSMITH_API_KEY)")

GOOGLE_API_KEY: set
LangSmith tracing: off (no valid LANGSMITH_API_KEY)


## Summarize messages


In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [4]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is exploring fictional, playful information about the moon, specifically inquiring about its capital, weather, population of cheese miners, and potential labor strikes.\n\n## SUMMARY\n\n- Established that the capital of the moon is Lunapolis.\n- Documented the current weather in Lunapolis (clear skies, high of 120C, low of -100C).\n- Noted a population of 100,000 cheese miners living in Lunapolis.\n- Discussed the likelihood of a cheese miners' union strike due to unhappiness with the new president.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\n- Await further questions or prompts from the user regarding Lunapolis, its inhabitants, or other fictional moon-related topics.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='e31ccb76-8635-48b8-b52e-4af7c84bee74'),
              HumanMessage(content="If you were Lunapolis' new president how would you r

In [5]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user is exploring fictional, playful information about the moon, specifically inquiring about its capital, weather, population of cheese miners, and potential labor strikes.

## SUMMARY

- Established that the capital of the moon is Lunapolis.
- Documented the current weather in Lunapolis (clear skies, high of 120C, low of -100C).
- Noted a population of 100,000 cheese miners living in Lunapolis.
- Discussed the likelihood of a cheese miners' union strike due to unhappiness with the new president.

## ARTIFACTS

None

## NEXT STEPS

- Await further questions or prompts from the user regarding Lunapolis, its inhabitants, or other fictional moon-related topics.


## Trim/delete messages


In [6]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [7]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [8]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='0952fa3a-a46e-4eef-8a57-a0a63025b425'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='5ec90fce-5fa5-408f-b648-80b1d8ecec47', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='e570d9a3-405b-4f22-986d-6e67141d4199'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='bcbe9b52-8f81-44d0-92be-f23149b8beab', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='70fa7711-5180-45eb-820c-7627e0c89a04'),
              AIMessage(content=[{'type': 'text', 'text': "It feels room temperature, neither h

In [9]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': "It feels room temperature, neither hot nor cold. \n\nSince it's plugged in and you mentioned earlier that it's turned on at the wall, are there any lights or indicators glowing on the device itself (like a charging light or power LED)?", 'extras': {'signature': 'EmAKXgFpFH0TBVaCgoTVM6kWpMGMAejVC+dyKZa+cqhRJHsuAfdnXaXYyCQNIAYMXLqf8BVd5cOEtB/Vsp8GG2XiQ67SIw3GLv69s1+0QWhnntfIi0hMp9wSR51L9R9G0yI='}}]


## Your task

> - Make a conversation too long on purpose, then add summarisation and watch where it cuts.
> - Write one before agent hook that removes something from the state, then prove it worked by asking the agent a question only the removed content could answer.
> - Look at the token counts before and after. That number is your bill.

### My response — by Kassouar Fatima 🇩🇿 — Module 3 project

All three tasks ran here, and each one printed its own evidence.

**1 · A conversation too long on purpose, then watch where it cuts.**
Nine turns of a moon conversation were fed to an agent with
`SummarizationMiddleware(model=..., trigger=("tokens", 100), keep=("messages", 1))`. It fired, and
the first message in state became **`Here is a summary of the conversation to date:`** — the whole
`## SESSION INTENT` / `## SUMMARY` / `## ARTIFACTS` / `## NEXT STEPS` block. **What got cut:** every
old turn, all of them. **What was kept:** `keep=("messages", 1)` — only the newest message. The
summary is built from the turns it then discards, so the intent, the artefacts and the next steps
survive instead of the raw chatter.

**2 · One `before_agent` hook that removes state — then prove it.**
`trim_messages` collects every `ToolMessage` in `state["messages"]` and returns
`{"messages": [RemoveMessage(id=m.id) for m in tool_messages]}` — deleted by id, before the model
runs. The proof is a question only the removed content could answer: the conversation I passed in
contains `ToolMessage(content="temp=42C voltage=2.9v … greeble complete.")`, and the last message
asks *"What's the temperature of the device?"*. The answer came back:

> **"It feels room temperature, neither hot nor cold."**

It did not say 42C — because `temp=42C` was gone from state before the model ever saw it. The hook
did exactly what it claims, and the wrongness of the answer is the proof.

**3 · Look at the token counts. That is your bill.**
Both runs print `usage_metadata` on the final message:

| run | input | output | **total** |
|---|---|---|---|
| summarise — moon conversation | 195 | 460 | **655** |
| trim — device conversation | 56 | 52 | **108** |

They are two different conversations, so read each as the bill *for that path*, not as a strict
A/B. The number to watch is `input_tokens`, because that is the side both middlewares actually cut:
**195 → 56**. The 460 output tokens on the first run are the summary being written — summarising
is not free, it is one extra model call whose cost shows up on the output side. You pay either way;
the question is whether you pay for raw history or for a summary you can keep using.

## Summary — main functions & parameters

| Function / parameter | What it does |
|---|---|
| `SummarizationMiddleware(model=..., trigger=("tokens", 100), keep=("messages", 1))` | collapse history once it crosses 100 tokens, keeping only the newest message |
| `create_agent(model=..., checkpointer=InMemorySaver(), middleware=[...])` | the checkpointer is what makes state worth summarising or trimming in the first place |
| `@before_agent trim_messages(state, runtime)` | runs before every model call; returns `{"messages": [RemoveMessage(id=m.id) ...]}` |
| `RemoveMessage(id=...)` | deletes a message from state **by id** — never by index |
| `ToolMessage` | the message type this hook filters out of `state["messages"]` |
| `agent.invoke(..., {"configurable": {"thread_id": "1"}})` | one thread per conversation — the checkpointer needs it |
| `response["messages"][-1].usage_metadata` | `input_tokens` / `output_tokens` / `total_tokens` — the bill: **655** (summarise run) vs **108** (trim run) |

---
*by Kassouar Fatima 🇩🇿 — Module 3 project*